<a href="https://colab.research.google.com/github/ukharel/Intermediate-Python/blob/master/Pydantic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In production level application, pydantic is used for data validator and type checking. Since python is dynamically typed which means you can create a variable and put whatever datatype value there, it can accepts it. But when you mistakely put 'str' instead of 'int' for production ml model, after 4 hours of training it will throw type error, wasting huge amount of cost which is nightmare for the model developer. so it is necessary to validate the data type so that the error won't occur.

In [9]:
def student_data(name,age:int) -> int:
  return f'{name} : {age}'


In [10]:
# Even though you have mention the data type should be int, and if you enter str , python will accept that because it is dyanamically typed.
student_data('Ujjwal','Twenty-five')

'Ujjwal : Twenty-five'

In [11]:
# To strictly accept whether the data type should be int or str, we need to write a type check code for that
def student_data(name:str,age:int):
  if type(name) == str and type(age) == int:
    print(name)
    print(age)
    print('inserted into the database....')
  else:
    raise TypeError('Data type is not matched.')

student_data('Ujjwal',30)



Ujjwal
30
inserted into the database....


In [12]:
# Here the issue is if you have to create another function for instance, update function, you have to manually again type check every data for safety which is very tediud.
# and also when you have to add some new parameter you have to manualyy update it everrywhere. To eliminate such type heavy and smooth data and type check, pydantic
# is used for large production level codebase.

# it is follows three major steps:
# 1) You have to define the input schema using class using basemodel
# 2) You have to create the dictionary of data and put it into variable. It is done so that pydantic can typecheck it before it is sent to the function. You also need to create the object fo the class
# 3) Now, you pass this object into the function, and do the action accordingly
from pydantic import BaseModel

class Student(BaseModel):
    name:str
    age:int

def student_data(stud:Student):
    print(stud.name)
    print(stud.age)
    print('data inserted into the database..')

student= {'name':'Ujjwal','age':'30'}
student1=Student(**student)
student_data(student1)

Ujjwal
30
data inserted into the database..


In [13]:
pip install "pydantic[email]"

Now let's do little complex data setting and let's see how pydantic handles the data. I will copy the code from above and extend to it.


In [14]:
from pydantic import BaseModel,EmailStr,AnyUrl,Field, field_validator
from typing import List, Dict, Optional,Annotated

class Student(BaseModel):
    name: Annotated[str,Field(max_length=20)]
    age:int = Field(gt=0,lt=20)
    email: EmailStr
    address: Optional[str] = None
    Attendance: bool
    subject: List[str]
    contact: Dict[str,str]

    @field_validator('age')
    @classmethod
    def check_age(cls,value):
      if 0< value <30:
        return value
      else:
        raise ValueError('Age out of the bound')

    @field_validator('email')
    @classmethod
    def check_email(cls,value):
      value_domains= ['gmail.com','yahoo.com','outlook.com']
      domain=value.split('@')[-1]
      if domain not in value_domains:
        raise ValueError('Email domain is not valid')
def student_data(stud:Student):
    print(stud.name)
    print(stud.age)
    print(stud.address)
    print('data inserted into the database..')

student= {'name':'Ujjwal','age':12, 'email':'abc@outlook.com','Attendance':True,'address':'Birtamode-5','subject':['Physics','Chemistry','English'],'contact':{'Phone_number':'124343'}}
student1=Student(**student)
student_data(student1)

Ujjwal
12
Birtamode-5
data inserted into the database..


Here, let's dig dive more using Data validator modules, field and field validator. I will copy above code and extend to it.

In [23]:
from pydantic import BaseModel,EmailStr,AnyUrl,Field, field_validator
from typing import List, Dict, Optional,Annotated

class Student(BaseModel):
    name: Annotated[str,Field(max_length=20)]
    age:int = Field(gt=0,lt=20)
    email: EmailStr
    address: Optional[str] = None
    Attendance: bool
    subject: List[str]
    contact: Dict[str,str]

    @field_validator('age',mode='after')
    @classmethod
    def check_age(cls,value):
      if 0< value <100:
        return value
      else:
        raise ValueError('Age out of the bound')

    @field_validator('email')
    @classmethod
    def check_email(cls,value):
      value_domains= ['gmail.com','yahoo.com','outlook.com']
      domain=value.split('@')[-1]
      if domain not in value_domains:
        raise ValueError('Email domain is not valid')
def student_data(stud:Student):
    print(stud.name)
    print(stud.age)
    print(stud.address)
    print('data inserted into the database..')

student= {'name':'Ujjwal','age':'12', 'email':'abc@outlook.com','Attendance':True,'address':'Birtamode-5','subject':['Physics','Chemistry','English'],'contact':{'Phone_number':'124343'}}
student1=Student(**student)
student_data(student1)

Ujjwal
12
Birtamode-5
data inserted into the database..


Now let's do model validator: what it does is it helps to make logic between two or more model input like age and contact. i will copy the code from above and extend to it.

In [25]:
from pydantic import BaseModel,EmailStr,AnyUrl,Field, field_validator, model_validator
from typing import List, Dict, Optional,Annotated

class Student(BaseModel):
    name: Annotated[str,Field(max_length=20)]
    age:int = Field(gt=0,lt=100)
    email: EmailStr
    address: Optional[str] = None
    Attendance: bool
    subject: List[str]
    contact: Dict[str,str]

    @model_validator(mode='after')
    def check_emergency_contact(cls,model):
      if model.age > 50 and 'emergency' not in model.contact:
        raise ValueError('Student greater than 50 years must have an emergency number')
      else:
        return model
    # @field_validator('age',mode='after')
    # @classmethod
    # def check_age(cls,value):
    #   if 0< value <30:
    #     return value
    #   else:
    #     raise ValueError('Age out of the bound')

    # @field_validator('email')
    # @classmethod
    # def check_email(cls,value):
    #   value_domains= ['gmail.com','yahoo.com','outlook.com']
    #   domain=value.split('@')[-1]
    #   if domain not in value_domains:
    #     raise ValueError('Email domain is not valid')
def student_data(stud:Student):
    print(stud.name)
    print(stud.age)
    print(stud.address)
    print('data inserted into the database..')

student= {'name':'Ujjwal','age':'65', 'email':'abc@outlook.com','Attendance':True,'address':'Birtamode-5','subject':['Physics','Chemistry','English'],'contact':{'Phone_number':'124343','emergency':'3455'}}
student1=Student(**student)
student_data(student1)

Ujjwal
65
Birtamode-5
data inserted into the database..


/tmp/ipykernel_2885/29033511.py:13: PydanticDeprecatedSince212: Using `@model_validator` with mode='after' on a classmethod is deprecated. Instead, use an instance method. See the documentation at https://docs.pydantic.dev/2.13/concepts/validators/#model-after-validator. Deprecated in Pydantic V2.12 to be removed in V3.0.
  @model_validator(mode='after')


In [29]:
# Nested modules
from pydantic import BaseModel

class Address(BaseModel):
  city:str
  state:str
  pincode:str
class Student(BaseModel):
  name:str
  age:int
  gender:str
  address: Address

addres= {'city':'Birtamode','state':'Koshi','pincode':'2323'}
address1= Address(**addres)
student_info={'name':'Ujjwal','age':30,'gender':'male','address':address1}
student = Student(**student_info)
print(student.address.pincode)

2323
